# BraTS 2021 3D UNet: Soft Label & Label Noise Research Pipeline

This notebook provides an end-to-end research testbed for studying **Label Noise Robustness**, **Continuous Soft Labels**, and **Boundary Uncertainty Calibration** in 3D Brain Tumor Segmentation (**MONAI 3D UNet**).

### Research Capabilities Included
1. **Static Per-Patient 3D Label Noise Simulation** (`src/noise_soft_labels.py`):
   - `'boundary'`: Smooth 3D morphological dilation/erosion simulating inter-observer over/under-segmentation.
   - `'flip'`: False-positive 3D ellipsoidal lesions and false-negative sub-region dropouts.
   - `'mixed'`: Combined boundary ambiguity and region-level annotation errors.
2. **Continuous Soft Label Formulations (`[0, 1]`)**:
   - `'label_smoothing'`: Uniform confidence calibration $(1 - \alpha) y + 0.5 \alpha$.
   - `'gaussian_boundary'`: Separable 3D Gaussian boundary softening.
   - `'sdf'`: Euclidean Signed Distance Function sigmoid softening $\sigma(\text{SDF}(x) / \tau)$.
3. **Soft-Label & Noise-Robust Losses** (`src/loss_metric.py`):
   - `soft_dice`, `soft_dice_bce`, `sce` (Symmetric Cross-Entropy), and `gce` (Generalized Cross-Entropy).
4. **Calibration & Uncertainty Evaluation**:
   - Evaluates against **clean held-out ground truth** using Dice (`WT`, `TC`, `ET`, `Mean`), **Expected Calibration Error (`ECE`)**, **Brier Score**, and **3D Predictive Entropy Maps**.

## 1. Environment Detection & Dynamic Project Setup

In [ ]:
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Environment detected: {'Google Colab' if IS_COLAB else 'Local Environment'}")

if IS_COLAB:
    !pip install -q monai nibabel scipy matplotlib scikit-learn tqdm

def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "config.py").is_file():
            return base.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from monai.inferers import sliding_window_inference

from src.config import get_softlabel_study_config, resolve_dataset_dir
from src.dataset import build_dataloaders, get_train_transforms
from src.eda import scan_brats_dataset, plot_patient_modalities
from src.loss_metric import build_loss_fn, BraTSMetric
from src.model import build_model, get_model_summary
from src.noise_soft_labels import (
    corrupt_label_3d,
    generate_soft_label_3d,
    compute_noise_corruption_stats,
)
from src.trainer import BraTSTrainer
from src.utils import (
    set_seed,
    load_checkpoint,
    plot_training_curves,
    visualize_noise_and_soft_labels,
    visualize_uncertainty_and_predictions,
)
from scripts.prepare_brats_mini import stream_brats_mini

print(f"Project Root: {PROJECT_ROOT}")

## 2. Dataset Discovery / Automatic `BraTS-Mini` Preparation

If `dataset/BraTS2021_Training_Data` is not yet populated, this cell automatically streams a lightweight `BraTS-Mini` cohort (default: 30 real BraTS 2021 3D MRI cases, ~285 MB) without downloading the full 13 GB archive.

In [ ]:
DATASET_DIR = Path(resolve_dataset_dir())
TARGET_MINI_CASES = 30

if not DATASET_DIR.exists() or len(scan_brats_dataset(str(DATASET_DIR))) == 0:
    print("Dataset not found locally — streaming BraTS-Mini subset...")
    stream_brats_mini(output_dir=DATASET_DIR, num_cases=TARGET_MINI_CASES)

records = scan_brats_dataset(str(DATASET_DIR))
print(f"Total verified BraTS 2021 cases available: {len(records)}")

## 3. Inspecting Synthetic Label Noise & Continuous Soft Labels

Visualize how boundary/region label noise corrupts expert annotations, and how **Gaussian Boundary** and **Signed Distance Function (SDF)** transforms convert sharp binary edges into calibrated probability transitions.

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    ConvertToMultiChannelBasedOnBratsClassesd, EnsureTyped, CropForegroundd
)

inspect_tf = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys="image"),
    ConvertToMultiChannelBasedOnBratsClassesd(keys="label"),
    EnsureTyped(keys=["image", "label"], dtype=[torch.float32, torch.float32]),
    CropForegroundd(keys=["image", "label"], source_key="image"),
])

sample = inspect_tf(records[0])
img_3d, clean_3d = sample["image"], sample["label"]

noisy_3d = corrupt_label_3d(
    clean_3d, image=img_3d, noise_type="mixed", noise_ratio=0.35, rng=np.random.RandomState(42)
)
gauss_soft_3d = generate_soft_label_3d(clean_3d, method="gaussian_boundary", sigma=1.8)
sdf_soft_3d = generate_soft_label_3d(clean_3d, method="sdf", sigma=1.8)

corruption = compute_noise_corruption_stats(clean_3d, noisy_3d)
print("Empirical Label Corruption Stats (Clean vs Noisy):", corruption)

fig = visualize_noise_and_soft_labels(
    flair_3d=img_3d[0].numpy(),
    clean_wt_3d=clean_3d[1].numpy(),
    noisy_wt_3d=noisy_3d[1].numpy(),
    gaussian_soft_wt_3d=gauss_soft_3d[1].numpy(),
    sdf_soft_wt_3d=sdf_soft_3d[1].numpy(),
    save_path="outputs/softlabel_study/notebook_noise_softlabel_demo.png",
)
plt.show()

## 4. Configure & Train 3D UNet Experiment Preset

Available presets in `get_softlabel_study_config()`:
- `'clean'`: Clean hard labels + `dice_ce` (Anchor Benchmark)
- `'noisy_baseline'`: Corrupted labels + `dice_ce` (Noise Degradation Check)
- `'noisy_soft_sdf'`: Corrupted labels + SDF Soft Labels + `soft_dice_bce`
- `'noisy_soft_gaussian'`: Corrupted labels + Gaussian Boundary Soft Labels + `soft_dice_bce`
- `'noisy_sce'`: Corrupted labels + Label Smoothing + Symmetric Cross-Entropy (`sce`)
- `'noisy_gce'`: Corrupted labels + Generalized Cross-Entropy (`gce`)

In [ ]:
PRESET = "noisy_soft_sdf"   # Change preset to compare methods
NOISE_RATIO = 0.35
MAX_SAMPLES = 20
EPOCHS = 15
VAL_INTERVAL = 3

config = get_softlabel_study_config(
    preset=PRESET,
    data_dir=str(DATASET_DIR),
    noise_ratio=NOISE_RATIO,
    roi_size=(80, 80, 80),
    lr=1e-3,
    max_epochs=EPOCHS,
    val_interval=VAL_INTERVAL,
)
set_seed(config.seed)

train_loader, val_loader, test_loader, train_files, val_files, test_files = build_dataloaders(
    data_dir=config.data_dir,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    batch_size=config.batch_size,
    num_workers=0,
    roi_size=config.roi_size,
    cache_rate=1.0,
    seed=config.seed,
    max_samples=MAX_SAMPLES,
    noise_type=config.noise_type,
    noise_ratio=config.noise_ratio,
    noise_radius=config.noise_radius,
    soft_label_method=config.soft_label_method,
    soft_alpha=config.soft_alpha,
    soft_sigma=config.soft_sigma,
    crop_eval_foreground=True,
)

model = build_model(config).to(config.device)
loss_fn = build_loss_fn(
    config.loss_type,
    sce_alpha=config.sce_alpha,
    sce_beta=config.sce_beta,
    gce_q=config.gce_q,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
metric = BraTSMetric(track_calibration=True)

trainer = BraTSTrainer(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    config=config,
    lr_scheduler=scheduler,
    metric=metric,
)

history = trainer.fit(train_loader, val_loader, max_epochs=EPOCHS, val_interval=VAL_INTERVAL)

## 5. Held-Out Test Evaluation & Boundary Uncertainty Visualization

In [ ]:
best_ckpt = os.path.join(config.checkpoint_dir, "best_metric_model.pth")
if os.path.exists(best_ckpt):
    load_checkpoint(best_ckpt, model, device=config.device)

test_scores = trainer.evaluate(test_loader, name=f"Held-Out Clean Test [{PRESET}]")
print("Test Scores & Calibration:", test_scores)

test_batch = next(iter(test_loader))
t_img = test_batch["image"].to(config.device)
t_gt = test_batch["label"].to(config.device)
with torch.no_grad():
    t_logits = sliding_window_inference(
        inputs=t_img, roi_size=config.roi_size, sw_batch_size=2,
        predictor=model, overlap=0.25, mode="gaussian"
    )
    t_probs = torch.sigmoid(t_logits)

fig_u = visualize_uncertainty_and_predictions(
    image=t_img[0].cpu().numpy(),
    clean_label=t_gt[0].cpu().numpy(),
    pred_probs=t_probs[0].cpu().numpy(),
    title_prefix=f"3D UNet ({PRESET})",
)
plt.show()